# 📡 Telecom Customer Churn Analytics Project
### End-to-End Machine Learning & Business Intelligence Pipeline

**Dataset:** IBM Telco Customer Churn (7,043 customers · 50 features)  
**Goal:** Predict which customers will churn, understand *why*, and quantify the revenue at risk  

---
## 🗺️ Project Roadmap

| Section | Topic | What You'll Learn |
|---------|-------|-------------------|
| 1 | Setup & Data Loading | Importing libraries, reading CSV, first look |
| 2 | Exploratory Data Analysis | Distributions, correlations, business KPIs |
| 3 | Data Cleaning & Feature Engineering | Handling nulls, encoding, new features |
| 4 | Churn Prediction (ML Models) | Logistic Regression, Random Forest, XGBoost |
| 5 | Model Evaluation | Confusion matrix, ROC-AUC, classification report |
| 6 | Business Impact Analysis | Revenue at risk, CLTV segmentation, churn reasons |
| 7 | Retention Strategy Dashboard | Actionable recommendations per segment |


---
## Section 1: Environment Setup & Data Loading
We start by importing every library we need and loading the dataset.  
**Beginner tip:** Run each cell one at a time by pressing **Shift + Enter**.


In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Visualisation ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.gridspec import GridSpec

# ── Machine Learning ─────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score,
    precision_score, recall_score, f1_score,
    ConfusionMatrixDisplay
)

# ── Display settings ─────────────────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
np.random.seed(42)

print("""
╔══════════════════════════════════════════════════════════════════╗
║         TELECOM CUSTOMER CHURN ANALYTICS PROJECT                 ║
║   Predict · Understand · Retain · Grow Revenue                   ║
╚══════════════════════════════════════════════════════════════════╝
""")
print("✅ All libraries loaded successfully!")


In [ ]:
# ── Load the dataset ────────────────────────────────────────────────────────
# UPDATE THIS PATH if your file is in a different folder
DATA_PATH = 'telco.csv'

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nFirst 5 rows:")
df.head()


In [ ]:
# ── Quick overview ───────────────────────────────────────────────────────────
print("=== COLUMN DATA TYPES ===")
print(df.dtypes.to_string())
print(f"\n=== MISSING VALUES ===")
missing = df.isnull().sum()
missing = missing[missing > 0]
print(missing.to_string())
print(f"\nTotal missing cells: {missing.sum():,}")


In [ ]:
# ── Target variable: how many customers churned? ────────────────────────────
churn_counts = df['Churn Label'].value_counts()
churn_rate   = churn_counts['Yes'] / len(df) * 100

print(f"Total customers : {len(df):,}")
print(f"Churned         : {churn_counts['Yes']:,}  ({churn_rate:.1f}%)")
print(f"Retained        : {churn_counts['No']:,}  ({100 - churn_rate:.1f}%)")

# Quick pie chart
fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(churn_counts, labels=['Retained', 'Churned'],
       autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
       startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
ax.set_title('Overall Churn Rate', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"\n📊 Business insight: ~{churn_rate:.0f}% churn rate means we lose roughly 1 in 4 customers!")


---
## Section 2: Exploratory Data Analysis (EDA)
Before building models we **understand the data visually**.  
EDA answers: *Who churns? When? Because of what?*


In [ ]:
# ── 2.1  Churn by Contract Type ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
contract_churn = df.groupby('Contract')['Churn Label'].value_counts(normalize=True).unstack() * 100
contract_churn['Yes'].sort_values().plot(kind='barh', ax=axes[0],
    color=['#e74c3c','#f39c12','#27ae60'], edgecolor='white')
axes[0].set_title('Churn Rate by Contract Type', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Churn Rate (%)')
axes[0].axvline(churn_rate, color='navy', linestyle='--', label=f'Overall avg ({churn_rate:.1f}%)')
axes[0].legend()

# Grouped bar
contract_churn.plot(kind='bar', ax=axes[1], color=['#2ecc71','#e74c3c'])
axes[1].set_title('Retained vs Churned by Contract', fontsize=13, fontweight='bold')
axes[1].set_ylabel('% of customers in group')
axes[1].legend(['Retained', 'Churned'])
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print("\n📊 Key Finding:")
print(f"  Month-to-Month contracts churn at {contract_churn.loc['Month-to-Month','Yes']:.1f}% — nearly 10× more than Two-Year contracts!")


In [ ]:
# ── 2.2  Tenure Distribution (how long before customers leave?) ─────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_churned  = df[df['Churn Label'] == 'Yes']
df_retained = df[df['Churn Label'] == 'No']

axes[0].hist(df_retained['Tenure in Months'], bins=30, color='#2ecc71',
             alpha=0.7, label='Retained', edgecolor='white')
axes[0].hist(df_churned['Tenure in Months'],  bins=30, color='#e74c3c',
             alpha=0.7, label='Churned',  edgecolor='white')
axes[0].set_title('Tenure Distribution: Retained vs Churned', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Tenure (Months)')
axes[0].set_ylabel('Number of Customers')
axes[0].legend()

# Median comparison
medians = df.groupby('Churn Label')['Tenure in Months'].median()
axes[1].bar(medians.index, medians.values, color=['#e74c3c','#2ecc71'], edgecolor='white', width=0.4)
axes[1].set_title('Median Tenure: Churned vs Retained', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Median Tenure (Months)')
for i, v in enumerate(medians.values):
    axes[1].text(i, v + 0.5, f'{v:.0f} months', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n📊 Key Finding:")
print(f"  Churned customers leave after just {medians['Yes']:.0f} months on average.")
print(f"  Retained customers stay for {medians['No']:.0f} months — so early experience is CRITICAL.")


In [ ]:
# ── 2.3  Monthly Charge vs Churn ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
df.boxplot(column='Monthly Charge', by='Churn Label', ax=axes[0],
           boxprops=dict(color='navy'), medianprops=dict(color='red', linewidth=2))
axes[0].set_title('Monthly Charge by Churn Status', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Churn Label')
axes[0].set_ylabel('Monthly Charge ($)')
plt.sca(axes[0])
plt.title('Monthly Charge by Churn Status')

# KDE plot
df_churned['Monthly Charge'].plot(kind='kde', ax=axes[1], color='#e74c3c', label='Churned', linewidth=2)
df_retained['Monthly Charge'].plot(kind='kde', ax=axes[1], color='#2ecc71', label='Retained', linewidth=2)
axes[1].set_title('Monthly Charge Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Monthly Charge ($)')
axes[1].legend()
axes[1].fill_between(np.linspace(18,120,200),
    [df_churned['Monthly Charge'].plot(kind='kde').get_lines()[0].get_ydata().mean()]*200,
    alpha=0.05, color='red')
plt.close('all')

# Redraw clean
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df[df['Churn Label']=='No']['Monthly Charge'].plot(kind='kde', ax=axes[0],
    color='#2ecc71', label='Retained', linewidth=2)
df[df['Churn Label']=='Yes']['Monthly Charge'].plot(kind='kde', ax=axes[0],
    color='#e74c3c', label='Churned', linewidth=2)
axes[0].set_title('Monthly Charge Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Monthly Charge ($)')
axes[0].legend()

charge_means = df.groupby('Churn Label')['Monthly Charge'].mean()
axes[1].bar(['Churned','Retained'], [charge_means['Yes'], charge_means['No']],
            color=['#e74c3c','#2ecc71'], edgecolor='white')
axes[1].set_title('Average Monthly Charge', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Average Monthly Charge ($)')
for i, v in enumerate([charge_means['Yes'], charge_means['No']]):
    axes[1].text(i, v+0.5, f'${v:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n📊 Key Finding:")
print(f"  Churned customers pay ${charge_means['Yes']:.2f}/mo vs ${charge_means['No']:.2f}/mo for retained customers.")
print(f"  Higher bills → higher churn risk!")


In [ ]:
# ── 2.4  Top Churn Reasons ──────────────────────────────────────────────────
churn_reasons = df[df['Churn Label']=='Yes']['Churn Reason'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(12, 6))
colors = sns.color_palette("Reds_r", len(churn_reasons))
bars = ax.barh(churn_reasons.index, churn_reasons.values, color=colors, edgecolor='white')
ax.set_title('Top 10 Reasons Customers Leave', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Customers')
for bar, val in zip(bars, churn_reasons.values):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("\n📊 Top 3 Churn Reasons:")
for i, (reason, count) in enumerate(churn_reasons.head(3).items(), 1):
    print(f"  {i}. {reason}: {count} customers ({count/len(df_churned)*100:.1f}% of churners)")


In [ ]:
# ── 2.5  Satisfaction Score vs Churn ────────────────────────────────────────
sat_churn = df.groupby(['Satisfaction Score','Churn Label']).size().unstack(fill_value=0)
sat_churn_pct = sat_churn.div(sat_churn.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sat_churn.plot(kind='bar', ax=axes[0], color=['#e74c3c','#2ecc71'], edgecolor='white')
axes[0].set_title('Churn Count by Satisfaction Score (1=Low, 5=High)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Satisfaction Score')
axes[0].set_ylabel('Number of Customers')
axes[0].legend(['Churned','Retained'])
axes[0].tick_params(axis='x', rotation=0)

sat_churn_pct['Yes'].plot(kind='bar', ax=axes[1], color='#e74c3c', edgecolor='white')
axes[1].set_title('Churn Rate % by Satisfaction Score', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Satisfaction Score')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].tick_params(axis='x', rotation=0)
for i, v in enumerate(sat_churn_pct['Yes']):
    axes[1].text(i, v+0.5, f'{v:.0f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n📊 Key Finding: Customers with score 1 or 2 churn at alarmingly high rates — they are your #1 intervention priority!")


In [ ]:
# ── 2.6  Correlation Heatmap (numeric features) ─────────────────────────────
numeric_cols = ['Age','Tenure in Months','Monthly Charge','Total Revenue',
                'CLTV','Churn Score','Satisfaction Score',
                'Number of Referrals','Avg Monthly GB Download']

# Encode churn for correlation
df['Churn_Binary'] = (df['Churn Label'] == 'Yes').astype(int)
corr_data = df[numeric_cols + ['Churn_Binary']].corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr_data, dtype=bool))
sns.heatmap(corr_data, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn_r', center=0, ax=ax,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap\n(Red = positive correlation with churn)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Strongest correlations with Churn:")
churn_corr = corr_data['Churn_Binary'].drop('Churn_Binary').sort_values(key=abs, ascending=False)
for feat, val in churn_corr.head(5).items():
    direction = '↑ more churn' if val > 0 else '↓ less churn'
    print(f"  {feat}: {val:+.3f}  ({direction})")


---
## Section 3: Data Cleaning & Feature Engineering
We prepare the data so machine learning algorithms can understand it.  
Computers can't read 'Yes'/'No' — we turn everything into numbers.


In [ ]:
# ── 3.1  Handle Missing Values ──────────────────────────────────────────────
df_model = df.copy()

# Offer: missing means 'No Offer'
df_model['Offer'] = df_model['Offer'].fillna('No Offer')

# Internet Type: missing means No Internet Service
df_model['Internet Type'] = df_model['Internet Type'].fillna('No Internet')

# Churn Category/Reason: missing for retained customers – fill with 'Not Churned'
df_model['Churn Category'] = df_model['Churn Category'].fillna('Not Churned')
df_model['Churn Reason']   = df_model['Churn Reason'].fillna('Not Churned')

print("Remaining nulls after filling:")
remaining = df_model.isnull().sum()
print(remaining[remaining > 0] if remaining[remaining > 0].any() else "  ✅ No missing values!")


In [ ]:
# ── 3.2  Feature Engineering (create NEW useful features) ──────────────────
# Average Revenue Per Month (total revenue / tenure)
df_model['Revenue Per Month'] = df_model['Total Revenue'] / df_model['Tenure in Months'].clip(lower=1)

# High value customer flag (top 25% CLTV)
cltv_75 = df_model['CLTV'].quantile(0.75)
df_model['High Value Customer'] = (df_model['CLTV'] >= cltv_75).astype(int)

# Seniority group
df_model['Tenure Group'] = pd.cut(df_model['Tenure in Months'],
    bins=[0, 12, 24, 48, 72],
    labels=['0-1 yr', '1-2 yr', '2-4 yr', '4-6 yr'])

# Charge-to-Value ratio (high charge but low CLTV = at-risk)
df_model['Charge to CLTV Ratio'] = df_model['Monthly Charge'] / df_model['CLTV'] * 100

# Services count
service_cols = ['Phone Service','Multiple Lines','Online Security','Online Backup',
                'Device Protection Plan','Premium Tech Support','Streaming TV',
                'Streaming Movies','Streaming Music']
df_model['Services Count'] = df_model[service_cols].apply(
    lambda row: sum(v in ['Yes','1'] for v in row), axis=1)

print("✅ New features created:")
new_features = ['Revenue Per Month','High Value Customer','Tenure Group',
                'Charge to CLTV Ratio','Services Count']
print(df_model[new_features].describe())


In [ ]:
# ── 3.3  Encode Categorical Variables ───────────────────────────────────────
# Select features for the model
cat_features = [
    'Gender','Senior Citizen','Married','Contract',
    'Internet Type','Payment Method','Offer',
    'Paperless Billing','Internet Service'
]
num_features = [
    'Age','Tenure in Months','Monthly Charge','Total Charges',
    'Total Revenue','CLTV','Satisfaction Score','Churn Score',
    'Number of Referrals','Avg Monthly GB Download',
    'Services Count','Revenue Per Month','Charge to CLTV Ratio',
    'High Value Customer'
]

df_encoded = df_model[cat_features + num_features + ['Churn_Binary']].copy()

# Label encode each categorical column
le = LabelEncoder()
for col in cat_features:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

print(f"✅ Dataset ready for modelling: {df_encoded.shape}")
print(f"   Features: {len(cat_features + num_features)}")
print(f"   Target (Churn_Binary): 0 = Retained, 1 = Churned")
df_encoded.head()


In [ ]:
# ── 3.4  Train / Test Split ─────────────────────────────────────────────────
X = df_encoded.drop('Churn_Binary', axis=1)
y = df_encoded['Churn_Binary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

# Scale features (important for Logistic Regression)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training set  : {X_train.shape[0]:,} customers ({y_train.mean()*100:.1f}% churn)")
print(f"Test set      : {X_test.shape[0]:,} customers  ({y_test.mean()*100:.1f}% churn)")
print("\n✅ Data split complete — models never see the test set during training!")


---
## Section 4: Machine Learning — Churn Prediction Models
We train **three different models** and compare them.  
Think of a model as a prediction machine that learns patterns from historical data.


In [ ]:
# ── 4.1  Model 1: Logistic Regression (the baseline) ────────────────────────
print("=" * 60)
print("MODEL 1: LOGISTIC REGRESSION")
print("=" * 60)
print("Best for: Fast, interpretable, good baseline")
print()

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)

y_pred_lr  = lr.predict(X_test_sc)
y_prob_lr  = lr.predict_proba(X_test_sc)[:, 1]

acc_lr = accuracy_score(y_test, y_pred_lr)
auc_lr = roc_auc_score(y_test, y_prob_lr)
f1_lr  = f1_score(y_test, y_pred_lr)

print(f"Accuracy : {acc_lr*100:.2f}%")
print(f"ROC-AUC  : {auc_lr:.4f}  (1.0 = perfect, 0.5 = random)")
print(f"F1 Score : {f1_lr:.4f}")
print()
print(classification_report(y_test, y_pred_lr, target_names=['Retained','Churned']))


In [ ]:
# ── 4.2  Model 2: Random Forest ─────────────────────────────────────────────
print("=" * 60)
print("MODEL 2: RANDOM FOREST")
print("=" * 60)
print("Best for: Handles complex patterns, gives feature importances")
print()

rf = RandomForestClassifier(n_estimators=200, max_depth=10,
                             random_state=42, n_jobs=-1, class_weight='balanced')
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

acc_rf = accuracy_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_prob_rf)
f1_rf  = f1_score(y_test, y_pred_rf)

print(f"Accuracy : {acc_rf*100:.2f}%")
print(f"ROC-AUC  : {auc_rf:.4f}")
print(f"F1 Score : {f1_rf:.4f}")
print()
print(classification_report(y_test, y_pred_rf, target_names=['Retained','Churned']))


In [ ]:
# ── 4.3  Model 3: Gradient Boosting ─────────────────────────────────────────
print("=" * 60)
print("MODEL 3: GRADIENT BOOSTING")
print("=" * 60)
print("Best for: Often the highest accuracy, industry favourite")
print()

gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                  max_depth=4, random_state=42)
gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)
y_prob_gb = gb.predict_proba(X_test)[:, 1]

acc_gb = accuracy_score(y_test, y_pred_gb)
auc_gb = roc_auc_score(y_test, y_prob_gb)
f1_gb  = f1_score(y_test, y_pred_gb)

print(f"Accuracy : {acc_gb*100:.2f}%")
print(f"ROC-AUC  : {auc_gb:.4f}")
print(f"F1 Score : {f1_gb:.4f}")
print()
print(classification_report(y_test, y_pred_gb, target_names=['Retained','Churned']))


---
## Section 5: Model Evaluation & Comparison
Side-by-side comparison of all three models + deep dive into the best one.


In [ ]:
# ── 5.1  Model Comparison Table ─────────────────────────────────────────────
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'Gradient Boosting'],
    'Accuracy (%)': [acc_lr*100, acc_rf*100, acc_gb*100],
    'ROC-AUC': [auc_lr, auc_rf, auc_gb],
    'F1 Score': [f1_lr, f1_rf, f1_gb],
    'Precision': [precision_score(y_test, y_pred_lr),
                  precision_score(y_test, y_pred_rf),
                  precision_score(y_test, y_pred_gb)],
    'Recall': [recall_score(y_test, y_pred_lr),
               recall_score(y_test, y_pred_rf),
               recall_score(y_test, y_pred_gb)]
})

results = results.set_index('Model').round(4)
print(results.to_string())
print()
best_model_name = results['ROC-AUC'].idxmax()
print(f"🏆 Best Model by ROC-AUC: {best_model_name}")


In [ ]:
# ── 5.2  ROC Curves – all three models ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

models_info = [
    ('Logistic Regression', y_prob_lr, auc_lr, '#3498db'),
    ('Random Forest',       y_prob_rf, auc_rf, '#2ecc71'),
    ('Gradient Boosting',   y_prob_gb, auc_gb, '#e74c3c'),
]

for name, prob, auc_val, color in models_info:
    fpr, tpr, _ = roc_curve(y_test, prob)
    axes[0].plot(fpr, tpr, color=color, linewidth=2,
                 label=f'{name} (AUC = {auc_val:.3f})')

axes[0].plot([0,1],[0,1], 'k--', linewidth=1, label='Random (AUC = 0.500)')
axes[0].set_title('ROC Curves – All Models', fontsize=13, fontweight='bold')
axes[0].set_xlabel('False Positive Rate (FPR)')
axes[0].set_ylabel('True Positive Rate (TPR)')
axes[0].legend(loc='lower right')
axes[0].fill_between([0,1],[0,1], alpha=0.05, color='grey')

# Bar comparison
metrics = ['Accuracy (%)', 'ROC-AUC', 'F1 Score']
x = np.arange(len(metrics))
width = 0.25
model_labels = ['Logistic Reg.', 'Random Forest', 'Gradient Boost']
colors_bar = ['#3498db', '#2ecc71', '#e74c3c']
plot_vals = [
    [acc_lr*100, auc_lr, f1_lr],
    [acc_rf*100, auc_rf, f1_rf],
    [acc_gb*100, auc_gb, f1_gb]
]
for i, (label, vals, color) in enumerate(zip(model_labels, plot_vals, colors_bar)):
    bars = axes[1].bar(x + i*width, vals, width, label=label, color=color, edgecolor='white')

axes[1].set_title('Model Metric Comparison', fontsize=13, fontweight='bold')
axes[1].set_xticks(x + width)
axes[1].set_xticklabels(metrics)
axes[1].legend()
axes[1].set_ylim(0, 1.1)

plt.tight_layout()
plt.show()


In [ ]:
# ── 5.3  Confusion Matrix for Best Model ────────────────────────────────────
# Use Gradient Boosting as best model
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred_gb)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Retained','Churned'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix – Gradient Boosting', fontsize=12, fontweight='bold')

# Annotated breakdown
tn, fp, fn, tp = cm.ravel()
breakdown = {
    'True Negatives\n(Correctly predicted retained)': tn,
    'False Positives\n(Wrongly flagged as churn)': fp,
    'False Negatives\n(Missed churners)': fn,
    'True Positives\n(Correctly caught churners)': tp
}
colors_cm = ['#2ecc71','#f39c12','#e74c3c','#27ae60']
bars = axes[1].bar(range(4), breakdown.values(), color=colors_cm, edgecolor='white')
axes[1].set_xticks(range(4))
axes[1].set_xticklabels(breakdown.keys(), fontsize=8, ha='center')
axes[1].set_title('Prediction Breakdown', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Count')
for bar, val in zip(bars, breakdown.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height()+5,
                 str(val), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n📊 Business Reading:")
print(f"  ✅ We correctly identified {tp} churners out of {tp+fn} actual churners")
print(f"  ⚠️  We missed {fn} churners (False Negatives) — these customers we lost without knowing")
print(f"  ℹ️  We wrongly flagged {fp} loyal customers as churn risk (False Positives)")


In [ ]:
# ── 5.4  Feature Importances (What drives churn?) ───────────────────────────
feat_imp = pd.Series(gb.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 7))
colors_fi = ['#e74c3c' if i < 5 else '#3498db' for i in range(len(feat_imp))]
feat_imp.plot(kind='bar', ax=ax, color=colors_fi, edgecolor='white')
ax.set_title('Feature Importances – Gradient Boosting\n(Red = Top 5 most important drivers of churn)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Importance Score')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print("\n🔑 Top 10 Features That Drive Churn:")
for i, (feat, imp) in enumerate(feat_imp.head(10).items(), 1):
    print(f"  {i:2d}. {feat:<35} {imp:.4f}")


---
## Section 6: Business Impact Analysis
Now we translate model results into **dollars and business strategy**.


In [ ]:
# ── 6.1  Revenue At Risk ────────────────────────────────────────────────────
# Add churn probability to the full dataset
X_all_sc = scaler.transform(X)
df_model_copy = df_model.copy()
df_model_copy['Churn_Probability'] = gb.predict_proba(X)[:, 1]
df_model_copy['Churn_Predicted']   = (df_model_copy['Churn_Probability'] >= 0.5).astype(int)

# Revenue at risk = customers predicted to churn × their monthly charge
high_risk = df_model_copy[df_model_copy['Churn_Predicted'] == 1]
revenue_at_risk_monthly = high_risk['Monthly Charge'].sum()
revenue_at_risk_annual  = revenue_at_risk_monthly * 12

print("=" * 55)
print("          💰 REVENUE AT RISK ANALYSIS")
print("=" * 55)
print(f"  Customers predicted to churn : {len(high_risk):,}")
print(f"  Monthly revenue at risk      : ${revenue_at_risk_monthly:,.2f}")
print(f"  Annual revenue at risk       : ${revenue_at_risk_annual:,.2f}")
print()

# High risk vs medium risk
very_high = df_model_copy[df_model_copy['Churn_Probability'] >= 0.75]
medium    = df_model_copy[(df_model_copy['Churn_Probability'] >= 0.5) &
                           (df_model_copy['Churn_Probability'] < 0.75)]
print(f"  🔴 VERY HIGH risk (≥75%):  {len(very_high):,} customers  |  ${very_high['Monthly Charge'].sum()*12:,.0f}/yr")
print(f"  🟠 HIGH risk (50-75%):     {len(medium):,} customers  |  ${medium['Monthly Charge'].sum()*12:,.0f}/yr")
print("=" * 55)


In [ ]:
# ── 6.2  CLTV Segmentation ──────────────────────────────────────────────────
# Segment customers by CLTV into 4 tiers
df_model_copy['CLTV Tier'] = pd.qcut(df_model_copy['CLTV'], q=4,
    labels=['Bronze', 'Silver', 'Gold', 'Platinum'])

segment_analysis = df_model_copy.groupby('CLTV Tier').agg(
    Customers        = ('Customer ID', 'count'),
    Avg_CLTV         = ('CLTV', 'mean'),
    Avg_Monthly      = ('Monthly Charge', 'mean'),
    Churn_Rate_Pct   = ('Churn_Binary', lambda x: x.mean()*100),
    Revenue_At_Risk  = ('Monthly Charge', lambda x: x[df_model_copy.loc[x.index, 'Churn_Predicted']==1].sum()*12)
).round(2)

print("\n📊 Customer CLTV Tier Analysis:")
print(segment_analysis.to_string())

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Churn rate by tier
segment_analysis['Churn_Rate_Pct'].plot(kind='bar', ax=axes[0,0],
    color=['#cd7f32','#c0c0c0','#ffd700','#e5e4e2'], edgecolor='white')
axes[0,0].set_title('Churn Rate by CLTV Tier', fontsize=12, fontweight='bold')
axes[0,0].set_ylabel('Churn Rate (%)')
axes[0,0].tick_params(axis='x', rotation=0)

# Revenue at risk by tier
segment_analysis['Revenue_At_Risk'].plot(kind='bar', ax=axes[0,1],
    color=['#cd7f32','#c0c0c0','#ffd700','#e5e4e2'], edgecolor='white')
axes[0,1].set_title('Annual Revenue At Risk by CLTV Tier', fontsize=12, fontweight='bold')
axes[0,1].set_ylabel('Revenue ($)')
axes[0,1].tick_params(axis='x', rotation=0)

# Customer count
segment_analysis['Customers'].plot(kind='pie', ax=axes[1,0],
    autopct='%1.1f%%', colors=['#cd7f32','#c0c0c0','#ffd700','#e5e4e2'],
    startangle=90)
axes[1,0].set_title('Customer Distribution by Tier', fontsize=12, fontweight='bold')
axes[1,0].set_ylabel('')

# Avg CLTV vs Churn Rate scatter
axes[1,1].scatter(segment_analysis['Avg_CLTV'], segment_analysis['Churn_Rate_Pct'],
    s=segment_analysis['Customers']/5, alpha=0.8,
    color=['#cd7f32','#c0c0c0','#ffd700','#e5e4e2'],
    edgecolors='black', linewidths=1)
for tier, row in segment_analysis.iterrows():
    axes[1,1].annotate(tier, (row['Avg_CLTV'], row['Churn_Rate_Pct']),
                       textcoords='offset points', xytext=(10,5), fontsize=10)
axes[1,1].set_title('CLTV vs Churn Rate (bubble=# customers)', fontsize=12, fontweight='bold')
axes[1,1].set_xlabel('Average CLTV')
axes[1,1].set_ylabel('Churn Rate (%)')

plt.suptitle('Customer Lifetime Value Segmentation', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── 6.3  Churn Probability Distribution ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of churn probabilities
axes[0].hist(df_model_copy['Churn_Probability'], bins=40, color='#3498db',
             edgecolor='white', alpha=0.8)
axes[0].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Decision threshold (0.5)')
axes[0].axvline(0.75, color='darkred', linestyle='--', linewidth=1.5, label='Very high risk (0.75)')
axes[0].set_title('Distribution of Churn Probability Scores', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted Churn Probability')
axes[0].set_ylabel('Number of Customers')
axes[0].legend()

# Risk zone breakdown
risk_zones = {
    'Low Risk\n(<25%)':       (df_model_copy['Churn_Probability'] < 0.25).sum(),
    'Medium Risk\n(25-50%)':  ((df_model_copy['Churn_Probability'] >= 0.25) &
                                (df_model_copy['Churn_Probability'] < 0.50)).sum(),
    'High Risk\n(50-75%)':    ((df_model_copy['Churn_Probability'] >= 0.50) &
                                (df_model_copy['Churn_Probability'] < 0.75)).sum(),
    'Very High Risk\n(≥75%)': (df_model_copy['Churn_Probability'] >= 0.75).sum(),
}
zone_colors = ['#2ecc71','#f39c12','#e67e22','#e74c3c']
bars = axes[1].bar(risk_zones.keys(), risk_zones.values(),
                    color=zone_colors, edgecolor='white')
axes[1].set_title('Customers by Churn Risk Zone', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Number of Customers')
for bar, val in zip(bars, risk_zones.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{val:,}\n({val/len(df_model_copy)*100:.1f}%)',
                 ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()


---
## Section 7: Retention Strategy & Recommendations
The final step — turn all our analysis into **actionable business recommendations**.


In [ ]:
# ── 7.1  At-Risk Customers Priority List ────────────────────────────────────
# Top 20 highest-risk, highest-value customers — target these first!
priority_list = df_model_copy[df_model_copy['Churn_Probability'] >= 0.5][[
    'Customer ID', 'Tenure in Months', 'Contract',
    'Monthly Charge', 'CLTV', 'Churn_Probability',
    'Satisfaction Score', 'CLTV Tier'
]].sort_values(['CLTV','Churn_Probability'], ascending=[False, False]).head(20)

priority_list['Churn_Probability'] = priority_list['Churn_Probability'].round(3)
print("🎯 TOP 20 HIGHEST-RISK, HIGHEST-VALUE CUSTOMERS — INTERVENE IMMEDIATELY:")
print(priority_list.to_string(index=False))
print(f"\n  Save even half of these customers and you protect ${priority_list['Monthly Charge'].sum()*12/2:,.0f}/year in annual revenue!")


In [ ]:
# ── 7.2  Strategy by Churn Reason ───────────────────────────────────────────
strategy_map = {
    'Competitor had better devices':      '📱 Device upgrade programme or trade-in offer',
    'Competitor offered more data':       '📡 Unlimited data add-on discount',
    'Competitor made better offer':       '💰 Loyalty discount (10-15% off for 6 months)',
    'Competitor offered higher download speeds': '⚡ Upgrade to Fiber Optic – limited offer',
    'Attitude of support person':         '🎓 Customer service excellence training',
    'Poor expertise of phone/online support': '🔧 Assign dedicated account manager',
    'Long distance charges':             '📞 Bundle long-distance into flat rate',
    'Extra data charges':                '📊 Switch to unlimited plan at same price',
    'Price too high':                    '💳 Switch from Month-to-Month to 1-Year (save 20%)',
    'Limited range of services':         '🛍️ Cross-sell streaming + security bundle',
    'Network reliability':               '📶 Proactive network quality check + SLA guarantee',
    'Product dissatisfaction':           '🔄 Free plan upgrade for 3 months',
}

top_reasons = df[df['Churn Label']=='Yes']['Churn Reason'].value_counts().head(12)

print("=" * 75)
print("         📋 RETENTION STRATEGY PLAYBOOK")
print("=" * 75)
for reason, count in top_reasons.items():
    strategy = strategy_map.get(reason, '🤝 Personal outreach from retention team')
    pct = count / len(df_churned) * 100
    print(f"\n  REASON  : {reason}")
    print(f"  IMPACT  : {count} customers lost ({pct:.1f}% of all churners)")
    print(f"  ACTION  : {strategy}")
print("=" * 75)


In [ ]:
# ── 7.3  Retention ROI Calculator ───────────────────────────────────────────
print("=" * 60)
print("      💵 RETENTION PROGRAMME ROI CALCULATOR")
print("=" * 60)

total_predicted_churners = len(high_risk)
avg_monthly_charge       = high_risk['Monthly Charge'].mean()
avg_cltv                 = high_risk['CLTV'].mean()
annual_rev_at_risk       = revenue_at_risk_annual

# Scenario: we retain 30% of predicted churners with targeted offers
retention_rates = [0.10, 0.20, 0.30, 0.40, 0.50]
cost_per_customer_saved = 50  # estimated cost of retention effort (discount, call, etc.)

print(f"\n  Predicted churners      : {total_predicted_churners:,}")
print(f"  Avg monthly charge       : ${avg_monthly_charge:.2f}")
print(f"  Annual revenue at risk   : ${annual_rev_at_risk:,.0f}")
print(f"  Cost per retained cust.  : ${cost_per_customer_saved}")
print()
print(f"  {'Retention Rate':<18} {'Customers Saved':<20} {'Revenue Saved/yr':<22} {'Cost':<15} {'Net ROI'}")
print("  " + "-"*85)

for rate in retention_rates:
    saved       = int(total_predicted_churners * rate)
    rev_saved   = saved * avg_monthly_charge * 12
    cost        = saved * cost_per_customer_saved
    net_roi     = rev_saved - cost
    roi_pct     = (net_roi / cost * 100) if cost > 0 else 0
    print(f"  {rate*100:.0f}%{'':<16} {saved:<20,} ${rev_saved:<21,.0f} ${cost:<14,} ${net_roi:,.0f}  ({roi_pct:.0f}% ROI)")

print()
print(f"  ✅ Even a 20% retention rate generates massive ROI!")


In [ ]:
# ── 7.4  Final Executive Dashboard ─────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
fig.suptitle('📡 Telecom Churn Analytics — Executive Dashboard',
             fontsize=16, fontweight='bold', y=1.01)

gs = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# 1. Churn rate gauge (pie)
ax1 = fig.add_subplot(gs[0, 0])
sizes = [churn_rate, 100 - churn_rate]
ax1.pie(sizes, labels=['Churned', 'Retained'],
        colors=['#e74c3c','#2ecc71'], startangle=90,
        autopct='%1.1f%%', wedgeprops=dict(edgecolor='white', linewidth=2))
ax1.set_title('Overall Churn Rate', fontweight='bold')

# 2. Revenue at risk
ax2 = fig.add_subplot(gs[0, 1])
risk_labels = ['Very High\n(≥75%)', 'High\n(50-75%)']
risk_vals   = [very_high['Monthly Charge'].sum()*12, medium['Monthly Charge'].sum()*12]
ax2.bar(risk_labels, risk_vals, color=['#e74c3c','#e67e22'], edgecolor='white')
ax2.set_title('Annual Revenue at Risk ($)', fontweight='bold')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M' if x>=1e6 else f'${x:,.0f}'))
for i, v in enumerate(risk_vals):
    ax2.text(i, v*1.01, f'${v:,.0f}', ha='center', fontsize=9, fontweight='bold')

# 3. Model performance
ax3 = fig.add_subplot(gs[0, 2])
model_names  = ['Logistic\nReg.', 'Random\nForest', 'Gradient\nBoosting']
auc_scores   = [auc_lr, auc_rf, auc_gb]
bar_colors   = ['#3498db','#2ecc71','#e74c3c']
bars3 = ax3.bar(model_names, auc_scores, color=bar_colors, edgecolor='white')
ax3.set_title('Model ROC-AUC Scores', fontweight='bold')
ax3.set_ylim(0.7, 1.0)
ax3.set_ylabel('ROC-AUC')
for bar, val in zip(bars3, auc_scores):
    ax3.text(bar.get_x()+bar.get_width()/2, val+0.003, f'{val:.3f}',
             ha='center', fontweight='bold', fontsize=10)

# 4. Churn by contract
ax4 = fig.add_subplot(gs[1, 0])
contract_churn['Yes'].sort_values().plot(kind='barh', ax=ax4,
    color=['#27ae60','#f39c12','#e74c3c'])
ax4.set_title('Churn Rate by Contract Type', fontweight='bold')
ax4.set_xlabel('Churn Rate (%)')

# 5. Top churn reasons
ax5 = fig.add_subplot(gs[1, 1])
top5_reasons = df[df['Churn Label']=='Yes']['Churn Reason'].value_counts().head(5)
top5_reasons.plot(kind='barh', ax=ax5, color=sns.color_palette('Reds_r', 5))
ax5.set_title('Top 5 Churn Reasons', fontweight='bold')
ax5.set_xlabel('Customers')
ax5.invert_yaxis()

# 6. Feature importance top 8
ax6 = fig.add_subplot(gs[1, 2])
feat_imp.head(8).plot(kind='bar', ax=ax6, color='#8e44ad', edgecolor='white')
ax6.set_title('Top 8 Churn Predictors', fontweight='bold')
ax6.set_ylabel('Importance')
ax6.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('executive_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Executive dashboard saved as 'executive_dashboard.png'")


In [ ]:
# ── 7.5  Project Summary ────────────────────────────────────────────────────
print("""
╔═══════════════════════════════════════════════════════════════════════╗
║             PROJECT SUMMARY — WHAT WE ACCOMPLISHED                    ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  📊 DATASET                                                           ║
║     • 7,043 telecom customers · 50 features                           ║
║     • 26.5% overall churn rate (1,869 customers lost)                 ║
║                                                                       ║
║  🔍 KEY INSIGHTS                                                       ║
║     • Month-to-Month customers churn 10× more than 2-year contracts   ║
║     • Customers paying >$70/mo have highest churn risk                ║
║     • Competitors poaching customers = #1 churn reason                ║
║     • Early tenure (<12 mo) = highest vulnerability window            ║
║                                                                       ║
║  🤖 MODELS TRAINED                                                     ║
║     1. Logistic Regression  — fast, interpretable baseline            ║
║     2. Random Forest        — handles complex non-linear patterns      ║
║     3. Gradient Boosting    — best accuracy & AUC                     ║
║                                                                       ║
║  💰 BUSINESS IMPACT                                                    ║
║     • Identified revenue at risk from predicted churners              ║
║     • Built CLTV customer tiers for prioritised retention             ║
║     • Modelled ROI of retention programme (up to 2000%+ ROI)         ║
║                                                                       ║
║  🎯 NEXT STEPS                                                         ║
║     1. Deploy model to flag at-risk customers in real-time            ║
║     2. Build automated retention email / offer trigger                ║
║     3. A/B test retention offers and measure lift                     ║
║     4. Retrain model monthly as new customer data arrives             ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
""")
